# VieEdu Text-to-SQL - Unsloth Qwen 3B Colab

Notebook này fine-tune `direct_sql` SFT baseline bằng Unsloth trên Google Colab.

Luồng chạy:
1. Cài Unsloth theo kiểu Colab.
2. Clone repo và checkout branch chứa data/pipeline.
3. Convert training JSON sang SFT JSONL.
4. Train Qwen 2.5 Coder 3B 4-bit LoRA bằng Unsloth.
5. Chạy inference/evaluate nhanh.

Khuyến nghị runtime: GPU T4 hoặc tốt hơn. Chạy `Runtime > Change runtime type > GPU` trước khi Run all.

## 1. Install Unsloth

Cell này dựa trên pattern notebook chính thức của Unsloth cho Colab: dùng `uv`, cài `bitsandbytes`, `xformers`, `unsloth`, rồi pin `transformers`/`trl` tương thích. Nếu Colab yêu cầu restart runtime sau khi cài, restart rồi chạy lại từ cell này.

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:/usr/local/cuda/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")

!python -m pip install --upgrade pip uv

try:
    import numpy, PIL
    _numpy = f"numpy=={numpy.__version__}"
    _pil = f"pillow=={PIL.__version__}"
except Exception:
    _numpy = "numpy"
    _pil = "pillow"

!python -m pip uninstall -y bitsandbytes
!uv pip install --system --upgrade {_numpy} {_pil} torchvision xformers unsloth
!python -m pip install --no-cache-dir --force-reinstall bitsandbytes==0.46.1
!uv pip install --system transformers==4.56.2
!uv pip install --system --no-deps trl==0.22.2
!uv pip install --system datasets peft accelerate pyyaml sqlparse hf_transfer
!sudo ldconfig /usr/lib64-nvidia || true
!sudo ldconfig /usr/local/cuda/lib64 || true
!python -m bitsandbytes

In [ ]:
import torch, subprocess, os, sys
print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info(0)
    print(f"VRAM free/total: {free/1024**3:.2f}GB / {total/1024**3:.2f}GB")
    print("bf16 supported:", torch.cuda.is_bf16_supported())
!nvidia-smi

import bitsandbytes as bnb
print("bitsandbytes:", getattr(bnb, "__version__", "unknown"))

# Smoke test bitsandbytes before loading Unsloth. If this fails, restart runtime and rerun install cells.
from bitsandbytes.nn import Linear4bit
layer = Linear4bit(8, 4, bias=False, compute_dtype=torch.float16, quant_type="nf4").cuda()
x = torch.randn(2, 8, device="cuda", dtype=torch.float16)
with torch.no_grad():
    y = layer(x)
print("bitsandbytes 4-bit smoke test: OK", tuple(y.shape), y.dtype)

## 2. Clone Repo And Prepare Dataset

Nếu branch đã merge vào branch khác, đổi `REPO_BRANCH` bên dưới.

In [ ]:
from pathlib import Path
import os, subprocess, json

REPO_URL = "https://github.com/dnhkhoa/Vietnamese-Multi-Turn-Conversational-Text-to-SQL-For-University-Course-Registration-Systems.git"
REPO_BRANCH = "feat/setup-qwen3b-finetune"
WORKDIR = Path("/content/viedu-text2sql")

if WORKDIR.exists():
    !rm -rf {WORKDIR}
!git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {WORKDIR}

PROJECT_ROOT = WORKDIR
UNSLOTH_ROOT = PROJECT_ROOT / "viedu-unsloth-local"
DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "university_v02"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATASET_PATH:", DATASET_PATH, DATASET_PATH.exists())

In [ ]:
%cd /content/viedu-text2sql/viedu-unsloth-local
!python scripts/inspect_dataset.py --dataset_path "../data/processed/university_v02"
!python scripts/prepare_sft_dataset.py --dataset_path "../data/processed/university_v02" --mode direct_sql
!wc -l data/processed/train_sft.jsonl data/processed/dev_sft.jsonl data/processed/test_sft.jsonl

## 3. Training Config

Mặc định dùng `unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit`. Nếu Colab T4 bị OOM, đổi `MODEL_NAME` sang `unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit`, hoặc giảm `MAX_SEQ_LENGTH` xuống `512`.

In [ ]:
MODEL_NAME = "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 1024
LORA_R = 16
LORA_ALPHA = 32
LEARNING_RATE = 2e-4
NUM_TRAIN_EPOCHS = 1
SANITY_STEPS = 20
OUTPUT_DIR = "outputs/adapters/colab_unsloth_qwen3b_lora"
RUN_FULL_TRAINING = False  # đổi thành True sau khi sanity train pass
PUSH_TO_HUB = False
HF_REPO_ID = "dnhkhoa/viedu-qwen25-coder-3b-text2sql-lora"

## 3.1 Repair bitsandbytes If Needed

Chạy cell này nếu cell load Unsloth báo `bitsandbytes is not installed` hoặc `libnvJitLink.so.13`. Sau khi cell này pass, chạy lại cell load model.

In [ ]:
import os, torch
os.environ["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:/usr/local/cuda/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")
!python -m pip uninstall -y bitsandbytes
!python -m pip install --no-cache-dir --force-reinstall bitsandbytes==0.46.1
!sudo ldconfig /usr/lib64-nvidia || true
!sudo ldconfig /usr/local/cuda/lib64 || true
!python -m bitsandbytes

import bitsandbytes as bnb
from bitsandbytes.nn import Linear4bit
layer = Linear4bit(8, 4, bias=False, compute_dtype=torch.float16, quant_type="nf4").cuda()
x = torch.randn(2, 8, device="cuda", dtype=torch.float16)
with torch.no_grad():
    y = layer(x)
print("bitsandbytes repair OK", getattr(bnb, "__version__", "unknown"), tuple(y.shape))

## 4. Load Model With Unsloth

Import `unsloth` trước `transformers/trl` để Unsloth patch model đúng cách.

In [ ]:
from unsloth import FastLanguageModel
import torch

dtype = None
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = dtype,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = LORA_ALPHA,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 5. Load SFT Dataset

In [ ]:
from datasets import load_dataset

data_files = {
    "train": "data/processed/train_sft.jsonl",
    "validation": "data/processed/dev_sft.jsonl",
    "test": "data/processed/test_sft.jsonl",
}
dataset = load_dataset("json", data_files=data_files)
print(dataset)
print(dataset["train"][0]["text"][:1000])

## 6. Sanity Train 20 Steps

Luôn chạy sanity trước. Nếu OOM: giảm `MAX_SEQ_LENGTH`, giảm `LORA_R`, hoặc đổi sang 1.5B.

In [ ]:
from trl import SFTTrainer, SFTConfig
import torch

bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
fp16 = torch.cuda.is_available() and not bf16

sanity_args = SFTConfig(
    output_dir = OUTPUT_DIR + "/sanity_checkpoints",
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 8,
    max_steps = SANITY_STEPS,
    learning_rate = LEARNING_RATE,
    warmup_ratio = 0.03,
    weight_decay = 0.0,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    save_steps = SANITY_STEPS,
    save_total_limit = 1,
    packing = False,
    fp16 = fp16,
    bf16 = bf16,
    report_to = "none",
    seed = 3407,
)

sanity_trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    args = sanity_args,
)
sanity_result = sanity_trainer.train()
print(sanity_result.metrics)

## 7. Full Train

Sau khi sanity pass, đổi `RUN_FULL_TRAINING = True` ở config cell rồi chạy cell này. Notebook giữ eval tắt mặc định để tiết kiệm VRAM; đánh giá sẽ làm bằng script SQLite sau inference.

In [ ]:
if RUN_FULL_TRAINING:
    full_args = SFTConfig(
        output_dir = OUTPUT_DIR + "/checkpoints",
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LENGTH,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        num_train_epochs = NUM_TRAIN_EPOCHS,
        learning_rate = LEARNING_RATE,
        warmup_ratio = 0.03,
        weight_decay = 0.0,
        lr_scheduler_type = "cosine",
        optim = "adamw_8bit",
        logging_steps = 10,
        save_steps = 100,
        save_total_limit = 2,
        packing = False,
        fp16 = fp16,
        bf16 = bf16,
        report_to = "none",
        seed = 3407,
    )
    full_trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = dataset["train"],
        args = full_args,
    )
    full_result = full_trainer.train()
    print(full_result.metrics)
else:
    print("RUN_FULL_TRAINING is False. Set it to True after sanity training passes.")

## 8. Save Adapter

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved adapter:", OUTPUT_DIR)

if PUSH_TO_HUB:
    from huggingface_hub import notebook_login
    notebook_login()
    model.push_to_hub(HF_REPO_ID)
    tokenizer.push_to_hub(HF_REPO_ID)

## 9. Quick Inference

In [ ]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

sample = dataset["test"][0]
prompt = sample["text"].split("<|im_start|>assistant\n", 1)[0] + "<|im_start|>assistant\n"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens = 512,
    do_sample = False,
    pad_token_id = tokenizer.eos_token_id,
)
pred = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
print("PRED SQL:\n", pred)
print("GOLD SQL:\n", sample["gold_sql"])

## 10. Batch Inference + SQLite Eval

Cell này chạy một subset trước. Tăng `EVAL_LIMIT` hoặc đặt `None` để full test.

In [ ]:
import json, time, pathlib, re

EVAL_LIMIT = 20  # đổi None để chạy full test
pred_path = pathlib.Path("outputs/predictions/colab_unsloth_pred.jsonl")
pred_path.parent.mkdir(parents=True, exist_ok=True)

def strip_fences(text):
    text = text.strip()
    text = re.sub(r"^```(?:sql)?\\s*", "", text, flags=re.I)
    text = re.sub(r"\\s*```$", "", text)
    return text.strip()

rows = dataset["test"] if EVAL_LIMIT is None else dataset["test"].select(range(min(EVAL_LIMIT, len(dataset["test"]))))
with pred_path.open("w", encoding="utf-8") as f:
    for row in rows:
        prompt = row["text"].split("<|im_start|>assistant\n", 1)[0] + "<|im_start|>assistant\n"
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        start = time.perf_counter()
        outputs = model.generate(**inputs, max_new_tokens=512, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        latency_ms = round((time.perf_counter() - start) * 1000, 2)
        pred = strip_fences(tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
        f.write(json.dumps({
            "id": row["id"],
            "gold_sql": row["gold_sql"],
            "pred_sql": pred,
            "latency_ms": latency_ms,
            "metadata": row.get("metadata", {}),
        }, ensure_ascii=False) + "\n")

print("Saved:", pred_path)
!python scripts/evaluate_sql.py --pred_file outputs/predictions/colab_unsloth_pred.jsonl --gold_file data/processed/test_sft.jsonl --sqlite_db ../data/processed/university_v02/database/university_registration.sqlite

## 11. Download Adapter

Nếu không push Hugging Face Hub, zip adapter để tải về.

In [ ]:
!zip -r colab_unsloth_qwen3b_lora.zip {OUTPUT_DIR}
from google.colab import files
files.download("colab_unsloth_qwen3b_lora.zip")